# 00 — Verify the workshop environment
Run this before the first lab. It checks imports and makes one short model
call. Knowledge and Memory need a second embedding model configured later.



**Setup check before lesson 1.**


## Problem

A classroom lab fails when model setup is discovered mid-session.


## Concept

A smoke test separates package, chat-model, and embedding setup.

There are two model dependencies: a chat model for generation and an embedding model for semantic retrieval. Passing one check does not imply the other works.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

Every lesson adds one row to this system. The rows below the marker are still the same code from earlier lessons.


## Architecture of this lesson

```text
uv sync ──▶ Python kernel ──▶ CrewAI import
                                 ├──▶ chat LLM ──▶ short response
                                 └──▶ embedder ──▶ vector dimensions
```

uv environment → CrewAI import → configured LLM → short response. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Inspect package/model configuration, then run an independent chat and embedding smoke test

Inspect package/model configuration, then run an independent chat and embedding smoke test.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

import workshop  # configure project-local CrewAI storage before importing CrewAI
import crewai
from crewai.rag.embeddings.factory import build_embedder

from workshop.common import embedder_config, model

print("CrewAI:", crewai.__version__)
print("Embedding provider:", embedder_config()["provider"])
print("Model response:", model().call("Reply with exactly: workshop ready"))
embedding = build_embedder(embedder_config())(["workshop ready"])
print("Embedding dimensions:", len(embedding[0]))


## Failure mode to know about

A working chat response does not prove embedding works. Test both before teaching; a missing Ollama model or endpoint should be diagnosed here, not in the Knowledge lab.


## Production implication

Pin the model and embedding versions and test both on the actual network and laptop class. Record a fallback path before the event.


## Your turn

Run the cells and identify which dependency fails, if any.

**Checkpoint:** CrewAI imports, the model replies, and the embedding provider is explicit.
